In [1]:
import os

print(os.getcwd())

C:\Users\3568s\NuralRetail\notebooks


In [ ]:
# =========================================================
# 📦 NeuralRetail Inventory Intelligence System
# EOQ + Safety Stock + ABC-XYZ + Dead Stock Analytics
# =========================================================

In [1]:
# =========================================================
# 📦 Imports
# =========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler

In [9]:
# =========================================================
# 📂 Load Dataset
# =========================================================

df = pd.read_csv("../data/processed/cleaned_retail_data.csv")

df.head()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,totalprice,total_amount,...,GMM_Cluster,GMM_prob_cluster_0,GMM_prob_cluster_1,GMM_prob_cluster_2,GMM_prob_cluster_3,GMM_prob_cluster_4,GMM_prob_cluster_5,GMM_prob_cluster_6,GMM_prob_cluster_7,GMM_prob_cluster_8
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,30.0,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715
1,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,39.6,39.6,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,30.0,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715
3,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,30.6,30.6,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715
4,489435,22195,HEART MEASURING SPOONS LARGE,24,2009-12-01 07:46:00,1.65,13085.0,United Kingdom,39.6,39.6,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715


In [10]:
# =========================================================
# 🧹 Basic Cleaning
# =========================================================

df.dropna(inplace=True)

df = df[df["quantity"] > 0]

df = df[df["unitprice"] > 0]

df["invoicedate"] = pd.to_datetime(df["invoicedate"])

df.head()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,totalprice,total_amount,...,GMM_Cluster,GMM_prob_cluster_0,GMM_prob_cluster_1,GMM_prob_cluster_2,GMM_prob_cluster_3,GMM_prob_cluster_4,GMM_prob_cluster_5,GMM_prob_cluster_6,GMM_prob_cluster_7,GMM_prob_cluster_8
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,30.0,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715
1,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,39.6,39.6,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,30.0,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715
3,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,30.6,30.6,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715
4,489435,22195,HEART MEASURING SPOONS LARGE,24,2009-12-01 07:46:00,1.65,13085.0,United Kingdom,39.6,39.6,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715


In [11]:
# =========================================================
# 🧹 Basic Cleaning
# =========================================================

df.dropna(inplace=True)

df = df[df["quantity"] > 0]

df = df[df["unitprice"] > 0]

df["invoicedate"] = pd.to_datetime(df["invoicedate"])

df.head()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,totalprice,total_amount,...,GMM_Cluster,GMM_prob_cluster_0,GMM_prob_cluster_1,GMM_prob_cluster_2,GMM_prob_cluster_3,GMM_prob_cluster_4,GMM_prob_cluster_5,GMM_prob_cluster_6,GMM_prob_cluster_7,GMM_prob_cluster_8
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,30.0,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715
1,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,39.6,39.6,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,30.0,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715
3,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,30.6,30.6,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715
4,489435,22195,HEART MEASURING SPOONS LARGE,24,2009-12-01 07:46:00,1.65,13085.0,United Kingdom,39.6,39.6,...,8,0.017,0.0,0.0006,0.0,0.0,0.0,0.0,0.2109,0.7715


In [13]:
# =========================================================
# 📦 Inventory Aggregation
# =========================================================

df["revenue"] = df["quantity"] * df["unitprice"]

inventory_df = df.groupby("stockcode").agg({

    "quantity": ["sum", "mean", "std"],
    "revenue": "sum",
    "unitprice": "mean"

}).reset_index()


KeyError: "Column(s) ['revenue'] do not exist"

In [ ]:
# =========================================================
# 🏷️ Rename Columns
# =========================================================

inventory_df.columns = [

    "SKU",
    "TotalDemand",
    "AvgDemand",
    "DemandStd",
    "AnnualRevenue",
    "AvgUnitPrice"

]

inventory_df.head()

In [ ]:
# =========================================================
# 🧹 Handle Missing Std
# =========================================================

inventory_df["DemandStd"] = (
    inventory_df["DemandStd"]
    .fillna(0)
)

In [ ]:
# =========================================================
# 🏭 Simulated Inventory Features
# =========================================================

np.random.seed(42)

inventory_df["LeadTime"] = np.random.randint(1, 15, len(inventory_df))

inventory_df["OrderingCost"] = np.random.randint(
    50,
    500,
    len(inventory_df)
)

inventory_df["HoldingCost"] = np.random.randint(
    5,
    50,
    len(inventory_df)
)

inventory_df["CurrentStock"] = np.random.randint(
    10,
    500,
    len(inventory_df)
)

inventory_df.head()

In [ ]:
# =========================================================
# 📦 Economic Order Quantity (EOQ)
# =========================================================

inventory_df["EOQ"] = np.sqrt(

    (
        2 *
        inventory_df["TotalDemand"] *
        inventory_df["OrderingCost"]
    )
    /
    inventory_df["HoldingCost"]

)

inventory_df.head()

In [ ]:
# =========================================================
# 🛡️ Safety Stock
# =========================================================

Z = 1.65

inventory_df["SafetyStock"] = (

    Z *
    inventory_df["DemandStd"] *
    np.sqrt(inventory_df["LeadTime"])

)

inventory_df.head()

In [ ]:
# =========================================================
# 🔁 Reorder Point
# =========================================================

inventory_df["ReorderPoint"] = (

    inventory_df["AvgDemand"] *
    inventory_df["LeadTime"]

) + inventory_df["SafetyStock"]

inventory_df.head()

In [ ]:
# =========================================================
# 📊 ABC Classification
# =========================================================

inventory_df = inventory_df.sort_values(
    by="AnnualRevenue",
    ascending=False
)

inventory_df["CumulativeRevenue"] = (

    inventory_df["AnnualRevenue"].cumsum()
    /
    inventory_df["AnnualRevenue"].sum()

)

In [ ]:
def abc_class(x):

    if x <= 0.80:
        return "A"

    elif x <= 0.95:
        return "B"

    else:
        return "C"

In [ ]:
inventory_df["ABC_Class"] = (
    inventory_df["CumulativeRevenue"]
    .apply(abc_class)
)

inventory_df.head()

In [ ]:
# =========================================================
# 📈 Coefficient of Variation
# =========================================================

inventory_df["CV"] = (

    inventory_df["DemandStd"]
    /
    inventory_df["AvgDemand"]

)

inventory_df["CV"] = inventory_df["CV"].fillna(0)

In [ ]:
def xyz_class(cv):

    if cv <= 0.5:
        return "X"

    elif cv <= 1:
        return "Y"

    else:
        return "Z"

In [ ]:
inventory_df["XYZ_Class"] = (
    inventory_df["CV"]
    .apply(xyz_class)
)

inventory_df.head()

In [ ]:
# =========================================================
# 🧠 ABC-XYZ Matrix
# =========================================================

inventory_df["ABC_XYZ"] = (

    inventory_df["ABC_Class"]
    +
    inventory_df["XYZ_Class"]

)

inventory_df.head()

In [ ]:
# =========================================================
# ☠️ Dead Stock Analytics
# =========================================================

inventory_df["DaysSinceSale"] = np.random.randint(
    1,
    365,
    len(inventory_df)
)

inventory_df["TurnoverRatio"] = (

    inventory_df["TotalDemand"]
    /
    inventory_df["CurrentStock"]

)

In [ ]:
inventory_df["DeadStockScore"] = (

    inventory_df["DaysSinceSale"] * 0.4

    +

    inventory_df["HoldingCost"] * 0.3

    -

    inventory_df["TurnoverRatio"] * 0.3

)

inventory_df.head()

In [ ]:
# =========================================================
# 🚨 Dead Stock Products
# =========================================================

dead_stock = inventory_df.sort_values(
    by="DeadStockScore",
    ascending=False
)

dead_stock.head(10)

In [ ]:
# =========================================================
# 📊 ABC Distribution
# =========================================================

fig = px.histogram(

    inventory_df,
    x="ABC_Class",
    color="ABC_Class",
    title="ABC Inventory Classification"

)

fig.show()

In [ ]:
# =========================================================
# 📊 XYZ Distribution
# =========================================================

fig = px.histogram(

    inventory_df,
    x="XYZ_Class",
    color="XYZ_Class",
    title="XYZ Demand Variability"

)

fig.show()

In [ ]:
# =========================================================
# 📦 EOQ Distribution
# =========================================================

fig = px.box(

    inventory_df,
    y="EOQ",
    title="EOQ Distribution"

)

fig.show()

In [ ]:
# =========================================================
# 💾 Export Inventory Intelligence Dataset
# =========================================================

inventory_df.to_csv(
    "inventory_intelligence.csv",
    index=False
)

print("✅ Inventory Intelligence Dataset Saved")